In [1]:
import pandas as pd
import re
from datasets import load_dataset

def unify_fann_or_flop(df: pd.DataFrame):
    df = df.copy()

    df.rename(columns={
        'source': 'source',
        'title': 'poem_title',
        'tags': 'tags',
        'verse_count': 'verse_count',
        'author': 'poet_name',
        'era': 'poet_era',
        'meter': 'meter',
        'genre': 'genre',
        'id': 'poem_id',
        'explanation': 'overall_explanation',
        'poem_verses': 'poem_text',
        'raw_explanation': 'verses_explanation'
    }, inplace=True)

    def extract_verses(text):
        if not isinstance(text, str):
            return []

        # Split into blocks using verse number markers
        blocks = re.split(r'\n*\b\d+\b\n*', text)
        blocks = [block.strip() for block in blocks if block.strip()]

        verses = []
        for block in blocks:
            lines = [line.strip() for line in block.split('\n') if line.strip()]
            if len(lines) >= 2:
                verses.append('\t'.join(lines))
            elif len(lines) == 1:
                verses.append(lines[0])
        return verses

    def process_tags(tags: str):
        """
        Process the tags string to extract meter, genre, and poet_era.

        Returns:
            tuple: (meter, genre, poet_era) if valid, else None.
        """

        tags = str(tags).strip()
        if not isinstance(tags, str):
            return None

        # Remove surrounding brackets if present
        tags = tags.strip().lstrip('[').rstrip(']')

        # Extract values enclosed in single or double quotes
        parts = re.findall(r"'([^']+)'|\"([^\"]+)\"", tags)

        # Flatten the tuple pairs returned by re.findall
        parts = [p1 or p2 for p1, p2 in parts]

        if len(parts) != 3:
            return None

        return tuple(part.strip() for part in parts)

    valid_rows = []
    invalid_rows = []

    invalid_verses_count = 0
    invalid_tags_count = 0
    for idx, row in df.iterrows():
        verses = extract_verses(row['poem_text'])
        expected_count = pd.to_numeric(row['verse_count'], errors='coerce')

        # Validate verse count
        if pd.isna(expected_count) or expected_count != len(verses):
            invalid_rows.append(row)
            invalid_verses_count += 1
            continue

        # Process tags
        processed_tags = process_tags(row.get('tags', ''))
        if processed_tags is None:
            invalid_rows.append(row)
            invalid_tags_count += 1
            continue

        
        meter, genre, poet_era = processed_tags

        # Update row with processed data
        row['poem_text'] = '\n'.join(verses)
        row['meter'] = meter
        row['genre'] = genre
        row['poet_era'] = poet_era

        valid_rows.append(row)

    print(f"Invalid rows due to verse count mismatch: {invalid_verses_count}")
    print(f"Invalid rows due to tag processing failure: {invalid_tags_count}")
    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    invalid_df = pd.DataFrame(invalid_rows).reset_index(drop=True)

    print(len(valid_df), "valid rows")
    print(len(invalid_df), "invalid rows")

    # Remove the 'verse_count' column from the dataframe
    valid_df.drop(columns=['verse_count'], inplace=True, errors='ignore')

    return valid_df, invalid_df

In [4]:
ds = load_dataset("omkarthawakar/FannOrFlop")
split_name = "train" if "train" in ds.keys() else list(ds.keys())[0]
df_fann = ds[split_name].to_pandas()

# Convert all dataframe values to strings (use empty string for missing values)

In [5]:
valid,invalid = unify_fann_or_flop(df_fann)


Invalid rows due to verse count mismatch: 67
Invalid rows due to tag processing failure: 0
6917 valid rows
67 invalid rows
